In [ ]:
# Section 8.1: Metrics and simple report

try:
    from medpy.metric.binary import hd95
    _has_medpy = True
except Exception:
    _has_medpy = False


def compute_basic_metrics(gt_mask, pred_mask, spacing=(1.0,1.0,1.0)):
    # Convert to numpy if torch
    if hasattr(gt_mask, 'cpu'):
        gt = gt_mask.cpu().numpy()
    else:
        gt = gt_mask
    if hasattr(pred_mask, 'cpu'):
        pr = pred_mask.cpu().numpy()
    else:
        pr = pred_mask

    results = {}
    # Region masks
    def region_masks(m):
        ET = (m == 3)
        TC = (m == 1) | (m == 3)
        WT = (m > 0)
        return ET, TC, WT

    gt_ET, gt_TC, gt_WT = region_masks(gt)
    pr_ET, pr_TC, pr_WT = region_masks(pr)

    def dice(a, b):
        a = a.astype(bool); b = b.astype(bool)
        if a.sum() == 0 and b.sum() == 0:
            return 1.0
        if a.sum() == 0 and b.sum() > 0:
            return 0.0
        inter = np.logical_and(a,b).sum()
        return 2*inter / (a.sum() + b.sum() + 1e-8)

    results['Dice_ET'] = dice(gt_ET, pr_ET)
    results['Dice_TC'] = dice(gt_TC, pr_TC)
    results['Dice_WT'] = dice(gt_WT, pr_WT)

    # HD95 optionally
    if _has_medpy:
        try:
            results['HD95_ET'] = hd95(gt_ET.astype(np.uint8), pr_ET.astype(np.uint8), voxelspacing=spacing)
            results['HD95_TC'] = hd95(gt_TC.astype(np.uint8), pr_TC.astype(np.uint8), voxelspacing=spacing)
            results['HD95_WT'] = hd95(gt_WT.astype(np.uint8), pr_WT.astype(np.uint8), voxelspacing=spacing)
        except Exception as e:
            results['HD95_error'] = str(e)
    else:
        results['HD95_ET'] = None
        results['HD95_TC'] = None
        results['HD95_WT'] = None

    # Volumes
    voxel_vol = np.prod(spacing)
    results['Volume_GT_WT_mm3'] = gt_WT.sum() * voxel_vol
    results['Volume_PR_WT_mm3'] = pr_WT.sum() * voxel_vol

    return results

print('\n✅ Basic metrics utility ready')

# Section 8: Clinical Report Generation

This section aggregates quantitative metrics for each case and produces a compact clinical report (JSON and PDF).

Report contents:
- Segmentation Dice (ET, TC, WT)
- HD95 (if medpy installed)
- Volumetric measurements (mm^3) per region
- Relative tumor change vs prior scan (if provided)
- Visual summaries: slice overlays + 3D mesh images

PDF export requires `reportlab` or `matplotlib` export utilities. Here we produce JSON + PNG summary which can be combined into PDF externally.

In [ ]:
# Section 7.1: Orchestrator for single patient
import time

def run_pipeline_for_patient(patient_id, paths_dict, output_dir, config, resume=False):
    os.makedirs(output_dir, exist_ok=True)
    meta = {'patient_id': patient_id, 'start_time': datetime.utcnow().isoformat()}

    # 1. Missing modality detection
    missing_mask, num_missing = MissingModalityDetector.find_missing(paths_dict.get('t1'), paths_dict.get('t1ce'), paths_dict.get('t2'), paths_dict.get('flair'))
    meta['missing_mask'] = missing_mask.tolist()
    print(f"\n▶ Running pipeline for {patient_id} | Missing modalities: {num_missing}")

    # 2. Preprocessing (if enough modalities present)
    t0 = time.time()
    tensor, brain_mask, affine = preprocessor.preprocess_patient(paths_dict['t1'], paths_dict['t1ce'], paths_dict['t2'], paths_dict['flair'])
    meta['preprocessing_time_sec'] = time.time() - t0

    # 3. Synthesis if needed
    t0 = time.time()
    if MissingModalityDetector.should_synthesize(num_missing, threshold=1):
        try:
            tensor = synthesizer.synthesize(tensor, missing_mask)
            meta['synthesis_performed'] = True
        except Exception as e:
            print(f"⚠️ Synthesis failed: {e}")
            meta['synthesis_performed'] = False
    else:
        meta['synthesis_performed'] = False
    meta['synthesis_time_sec'] = time.time() - t0

    # 4. Segmentation
    t0 = time.time()
    # Load segmentation checkpoint state (dictionary) - user must instantiate model and pass state
    seg_state = SegmentationModelLoader.load(config['paths'].get('segmentation_model'), device=str(device))
    # NOTE: user must create UNet3D instance and load seg_state as in Section 5 example
    meta['segmentation_time_sec'] = time.time() - t0

    # 5. Post-process + 3D reconstruction
    t0 = time.time()
    # For demonstration we assume user has created `probs, pred` via run_segmentation_for_patient
    # Placeholder: create dummy pred from tensor center threshold
    pred_dummy = (tensor.cpu().numpy()[0,0] > 0).astype(np.uint8)  # simple example
    meshes = create_and_save_hierarchical_meshes(pred_dummy, affine, os.path.join(output_dir, f"{patient_id}"))
    meta['reconstruction_time_sec'] = time.time() - t0

    # 6. Save metadata
    meta['end_time'] = datetime.utcnow().isoformat()
    with open(os.path.join(output_dir, f"{patient_id}_meta.json"), 'w') as f:
        json.dump(meta, f, indent=2)
    print(f"\n✅ Pipeline finished for {patient_id}. Outputs saved in {output_dir}")
    return meta

# Example run (user must supply correct paths)
# patient_paths = {'t1':'/path/T1.nii.gz', 't1ce':'/path/T1ce.nii.gz', 't2':'/path/T2.nii.gz', 'flair':'/path/FLAIR.nii.gz'}
# run_pipeline_for_patient('Case_000', patient_paths, './output/Case_000', config.config)

print('\n✅ Pipeline orchestrator ready (demo mode)')

# Section 7: End-to-End Pipeline Execution

This section orchestrates the complete workflow for a single patient and for batch processing. It records timing for each stage and supports checkpointing to resume partially processed cases.

In [ ]:
# Section 6.2: Hierarchical Mesh Creation and Interactive View

def create_and_save_hierarchical_meshes(pred_mask, affine, output_prefix):
    """Create meshes for WT, TC, ET and save OBJ/STL"""
    # Region masks
    et_mask = (pred_mask == 3).astype(np.uint8)
    tc_mask = ((pred_mask == 1) | (pred_mask == 3)).astype(np.uint8)  # NCR + ET
    wt_mask = (pred_mask > 0).astype(np.uint8)

    meshes = {}
    for name, mask in [('WT', wt_mask), ('TC', tc_mask), ('ET', et_mask)]:
        if mask.sum() == 0:
            print(f"⚠️ No {name} voxels detected, skipping mesh")
            continue
        verts, faces = generate_mesh_from_mask(mask, spacing=(1.0,1.0,1.0), level=0.5)
        if verts is None:
            continue
        verts_s, faces_s = laplacian_smooth(verts, faces, iterations=config['reconstruction'].get('smooth_iterations',3), lamb=config['reconstruction'].get('smooth_lambda',0.5))
        obj_path = f"{output_prefix}_{name}.obj"
        export_obj(obj_path, verts_s, faces_s)
        meshes[name] = {'verts': verts_s, 'faces': faces_s, 'obj': obj_path}
    return meshes


def plot_interactive_meshes(meshes, title='Tumor Meshes', save_html=None):
    data = []
    color_map = {'WT':'green','TC':'yellow','ET':'red'}
    opacity_map = {'WT':0.35,'TC':0.6,'ET':0.9}
    for name, m in meshes.items():
        verts = m['verts']
        faces = m['faces']
        data.append(go.Mesh3d(x=verts[:,0], y=verts[:,1], z=verts[:,2], i=faces[:,0], j=faces[:,1], k=faces[:,2], color=color_map.get(name,'cyan'), opacity=opacity_map.get(name,0.5), name=name))
    fig = go.Figure(data=data)
    fig.update_layout(title=title, scene=dict(aspectmode='data'))
    if save_html:
        fig.write_html(save_html)
        print(f"✅ Saved interactive HTML: {save_html}")
    return fig

print('\n✅ 3D reconstruction and visualization utilities ready')

In [ ]:
# Section 6.1: Mesh generation utilities

def generate_mesh_from_mask(mask, spacing=(1.0,1.0,1.0), level=0.5, step_size=1):
    """Generate vertices and faces using marching cubes
    Args:
        mask: binary volume [H,W,D]
    Returns:
        verts: (N,3), faces: (M,3)
    """
    try:
        verts, faces, normals, values = measure.marching_cubes(mask, level=level, spacing=spacing, step_size=step_size)
        return verts, faces
    except Exception as e:
        print(f"⚠️ Marching cubes failed: {e}")
        return None, None


def laplacian_smooth(verts, faces, iterations=3, lamb=0.5):
    # Simple Laplacian smoothing
    V = verts.copy()
    adj = [[] for _ in range(len(V))]
    for f in faces:
        for i in range(3):
            a, b = f[i], f[(i+1)%3]
            adj[a].append(b)
            adj[b].append(a)
    for _ in range(iterations):
        newV = V.copy()
        for i in range(len(V)):
            neigh = adj[i]
            if len(neigh) == 0: continue
            avg = V[neigh].mean(axis=0)
            newV[i] = V[i] * (1 - lamb) + avg * lamb
        V = newV
    return V, faces


def export_obj(filepath, verts, faces):
    os.makedirs(Path(filepath).parent, exist_ok=True)
    with open(filepath, 'w') as f:
        for v in verts:
            f.write(f"v {v[0]} {v[1]} {v[2]}\n")
        for face in faces:
            # OBJ indices are 1-based
            f.write(f"f {face[0]+1} {face[1]+1} {face[2]+1}\n")
    print(f"✅ Exported OBJ: {filepath}")

print("\n✅ Mesh utilities ready")

# Section 6: 3D Mesh Reconstruction and Visualization

Convert label masks into smooth, watertight meshes for clinical use. Steps:
- Extract binary masks for ET, TC, WT from predicted label map
- Use marching cubes to get vertices and faces
- Laplacian smoothing (iterative)
- Optional decimation for rendering performance
- Export to OBJ/STL and create interactive Plotly view

**Region definitions (BraTS-style):**
- NCR (label=1), ED (label=2), ET (label=3)
- WT = NCR ∪ ED ∪ ET
- TC = NCR ∪ ET

In [ ]:
# Section 5.2: Full Segmentation Inference Example (requires user-provided model)
# User should replace `user_unet_model` with actual model instance and load state_dict

def run_segmentation_for_patient(model_state_dict, input_tensor, brain_mask, affine, output_dir):
    # 1. Create model instance - user must adapt to their UNet constructor
    try:
        # Attempt to import UNet3D from segmentation module
        from segmentaion_module.unet_eval import UNet3D  # placeholder - verify actual path
        model = UNet3D()
        model.load_state_dict(model_state_dict)
        print("✅ UNet model created and weights loaded")
    except Exception as e:
        print(f"⚠️ Could not instantiate UNet automatically: {e}")
        print("   Please create UNet3D instance manually and call SegmentationInferer with it.")
        return

    inferer = SegmentationInferer(model, device=str(device), roi_size=(config['segmentation']['patch_size'],)*3, overlap=config['segmentation']['overlap'])
    probs, pred = inferer.infer(input_tensor)

    # Post-process WT (whole tumor = classes 1,2,3)
    wt_mask = (pred > 0).astype(np.uint8)
    wt_mask = remove_small_components(wt_mask, min_size=100)

    # Save outputs
    os.makedirs(output_dir, exist_ok=True)
    prob_path = os.path.join(output_dir, 'probabilities.nii.gz')
    pred_path = os.path.join(output_dir, 'pred_mask.nii.gz')
    nib.save(nib.Nifti1Image(probs.astype(np.float32), affine), prob_path)
    nib.save(nib.Nifti1Image(pred.astype(np.uint8), affine), pred_path)
    print(f"✅ Saved probabilities: {prob_path}")
    print(f"✅ Saved prediction mask: {pred_path}")

    return probs, pred

print("\n✅ Segmentation example ready (requires model instance)")

In [ ]:
# Section 5.1: Sliding Window Inference Wrapper
try:
    from monai.inferers import sliding_window_inference
    _has_monai = True
except Exception:
    _has_monai = False

class SegmentationInferer:
    def __init__(self, model, device='cuda', roi_size=(64,64,64), sw_batch_size=2, overlap=0.5):
        self.model = model
        self.device = device
        self.roi_size = roi_size
        self.sw_batch_size = sw_batch_size
        self.overlap = overlap
        if _has_monai:
            print("✅ MONAI sliding_window_inference available")
        else:
            print("⚠️ MONAI not available → falling back to naive full-volume inference (might OOM)")
    
    def infer(self, input_tensor):
        """
        Run full-volume inference and return softmax probabilities and predicted label map
        Args:
            input_tensor: [1, C, H, W, D]
        Returns:
            probs: [C, H, W, D] (numpy)
            pred: [H, W, D] (numpy, integer labels)
        """
        self.model.to(self.device)
        self.model.eval()
        with torch.no_grad():
            if _has_monai:
                inputs = input_tensor.to(self.device)
                raw_logits = sliding_window_inference(
                    inputs=inputs,
                    roi_size=self.roi_size,
                    sw_batch_size=self.sw_batch_size,
                    predictor=lambda x: self._extract_logits(x),
                    overlap=self.overlap,
                    mode='gaussian'
                )
                logits = raw_logits.squeeze(0).cpu()
            else:
                # Fallback: run full model (may OOM)
                logits = self._extract_logits(input_tensor.to(self.device)).squeeze(0).cpu()

            probs = torch.softmax(logits, dim=0).numpy()  # [C, H, W, D]
            pred = np.argmax(probs, axis=0).astype(np.uint8)
            return probs, pred
    
    def _extract_logits(self, x):
        outputs = self.model(x)
        if isinstance(outputs, (tuple, list)):
            return outputs[0]
        return outputs

# Post-processing utilities

def remove_small_components(mask, min_size=100):
    labeled, num = ndimage.label(mask)
    if num == 0:
        return mask
    sizes = np.bincount(labeled.ravel())
    out = np.zeros_like(mask)
    for i in range(1, num+1):
        if sizes[i] >= min_size:
            out[labeled == i] = 1
    return out

print("\n✅ Segmentation inference utilities ready")

# Section 5: Segmentation Inference

**Inference Strategy:**
- Use MONAI/Torch sliding-window inference for memory efficiency
- Configurable `roi_size` (patch size) and `overlap`
- Softmax over logits to produce per-class probability maps
- Argmax to get final label map

**Post-processing Steps:**
- Remove small connected components (< min_size voxels)
- Fill small holes
- Enforce brain mask boundaries

**Outputs:**
- `pred_mask.nii.gz` (voxel labels)
- `probabilities.nii.gz` (float32, shape [C, H, W, D])
- `metrics.json` (Dice, volume per region)

In [ ]:
# Section 4.2: Placeholder Synthesis Inference Wrapper
# NOTE: The real DiffusionSynthesisModel inference API must be used here.
# This wrapper shows expected data flow and fallback behavior.
class SynthesisWrapper:
    def __init__(self, checkpoint_path=None, device='cuda'):
        self.checkpoint_path = checkpoint_path
        self.device = device
        self.model = None
        if checkpoint_path and os.path.exists(checkpoint_path):
            try:
                # Placeholder: user should import actual model class and load weights
                # from synthesis_module.model.architecture import DiffusionSynthesisModel
                # self.model = DiffusionSynthesisModel(...) and load state_dict
                print(f"⚠️ Synthesis model loading placeholder: {checkpoint_path}")
                self.loaded = False
            except Exception as e:
                print(f"❌ Failed to initialize synthesis model: {e}")
                self.loaded = False
        else:
            print("⚠️ No synthesis checkpoint provided or file missing")
            self.loaded = False

    def synthesize(self, input_tensor, missing_mask, sampling_steps=50):
        """
        Placeholder function demonstrating expected behavior:
        - input_tensor: [1, C, H, W, D] (C ≤ 4)
        - missing_mask: binary array length C indicating which channels to synthesize
        
        Returns:
            completed_tensor: [1, 4, H, W, D]
        """
        C = input_tensor.shape[1]
        if C == 4 and missing_mask.sum() == 0:
            return input_tensor

        # If model not loaded, fallback by repeating available modalities or zero-fill
        if not self.loaded:
            print("⚠️ Synthesis model not available → using fallback strategy")
            # Simple fallback: replace missing channels with mean of available channels
            avail = input_tensor[:, :C]
            mean_vol = avail.mean(dim=1, keepdim=True)
            # Compose final 4-channel tensor
            filled = []
            for i in range(4):
                if i < C and missing_mask[i] == 0:
                    filled.append(input_tensor[:, i:i+1])
                else:
                    filled.append(mean_vol)
            completed = torch.cat(filled, dim=1)
            return completed
        
        # If model loaded, call inference routine (user must implement exact API)
        raise NotImplementedError("Synthesis model inference not implemented in notebook placeholder")

# Initialize placeholder synthesis wrapper
synthesizer = SynthesisWrapper(config['paths'].get('synthesis_model'), device=str(device))
print("\n✅ Synthesis wrapper ready (placeholder)")

# Section 5: Segmentation Inference

**Segmentation Network: UNet3D**
- **Architecture**: 4-level encoder-decoder with residual connections + attention gates
- **Input**: 4-channel MRI volume (or 3 if synthesis skipped)
- **Output**: 4-class logits [Background, NCR, ED, ET]
- **Inference Method**: Sliding window (64³ patches, 50% overlap) for memory efficiency
- **Post-Processing**: 
  - Remove small connected components (<100 voxels)
  - Enforce boundaries with brain mask
  - Compute per-region (ET, TC, WT) and per-class (NCR, ED, ET) predictions

**Clinical Output Definitions:**
- **ET (Enhancing Tumor, Class 3)**: GD-enhancing tumor (high grade indicator)
- **ED (Peritumoral Edema, Class 2)**: Edema surrounding tumor (extent indicator)
- **NCR (Necrotic Core, Class 1)**: Non-enhancing core (size indicator)
- **WT (Whole Tumor)**: ED ∪ NCR ∪ ET (overall extent)
- **TC (Tumor Core)**: NCR ∪ ET (excludes edema)

In [ ]:
# Section 4.1: Missing Modality Detection
class MissingModalityDetector:
    """Detect missing modalities in patient input"""
    
    @staticmethod
    def find_missing(t1_path, t1ce_path, t2_path, flair_path):
        """
        Check which modalities are present/missing
        
        Returns:
            missing_mask: [T1, T1ce, T2, FLAIR] binary (0=present, 1=missing)
            num_missing: Integer count of missing modalities
        """
        paths = [t1_path, t1ce_path, t2_path, flair_path]
        present = [os.path.exists(p) if p else False for p in paths]
        missing = [1 - int(p) for p in present]
        
        return np.array(missing), sum(missing)
    
    @staticmethod
    def should_synthesize(num_missing, threshold=1):
        """Determine if synthesis should be triggered"""
        return num_missing >= threshold

# Test detector
print("✅ Missing Modality Detector initialized")

# Example usage:
# missing_mask, num_missing = MissingModalityDetector.find_missing(
#     't1.nii.gz', 't1ce.nii.gz', None, 'flair.nii.gz'
# )
# print(f"Missing: {num_missing} modalities - {missing_mask}")

# Section 4: Missing Modality Synthesis (Optional)

**When is synthesis needed:**
- One or more MRI modalities missing from acquisition
- Incomplete protocol due to patient movement, scanner issues
- Quality degradation in single modality

**Synthesis Architecture:**
- **Input**: Available modalities (e.g., 3 of 4 sequences) + missing modality indicator
- **Model**: Diffusion-based architecture with modality-specific encoders/decoders
- **Inference**: 50 diffusion sampling steps with deterministic DDIM schedule (eta=0.0)
- **Output**: Synthetic volume structurally consistent with available scans

**Graceful Degradation:**
- If synthesis fails: Skip synthesis, use available modalities only
- If synthesis model not loaded: Pipeline continues with 3-channel input
- System designed to work with 3-4 channels (not all modalities required)

In [ ]:
# Section 3.4: Complete Preprocessing Pipeline
class MultimodalPreprocessor:
    """End-to-end preprocessing for multi-modal MRI data"""
    
    def __init__(self, config):
        self.config = config
        self.normalizer = IntensityNormalizer(
            lower_percentile=config['preprocessing'].get('lower_percentile', 0.5),
            upper_percentile=config['preprocessing'].get('upper_percentile', 99.5)
        )
        self.loader = NIfTILoader()
    
    def preprocess_patient(self, t1_path, t1ce_path, t2_path, flair_path):
        """
        Complete preprocessing for one patient
        
        Args:
            t1_path, t1ce_path, t2_path, flair_path: Paths to modality files
        
        Returns:
            stacked_tensor: [1, 4, H, W, D] torch tensor
            brain_mask: [H, W, D] binary mask
            affine: Affine transformation matrix
        """
        print("\n🔄 Preprocessing Patient Data...")
        
        # 1. Load all modalities
        paths = [t1_path, t1ce_path, t2_path, flair_path]
        modalities = []
        affines = []
        
        for i, path in enumerate(paths):
            mod_name = ['T1', 'T1ce', 'T2', 'FLAIR'][i]
            vol, aff, _ = self.loader.load_volume(path)
            if vol is None:
                raise ValueError(f"Failed to load {mod_name}")
            modalities.append(vol)
            affines.append(aff)
            print(f"   ✅ Loaded {mod_name}: {vol.shape}")
        
        # 2. Generate common brain mask
        brain_mask = BrainMaskGenerator.common_mask_from_multimodal(modalities)
        print(f"   ✅ Generated brain mask: {brain_mask.sum()} voxels")
        
        # 3. Normalize each modality
        normalized_mods = []
        for i, mod in enumerate(modalities):
            normalized = self.normalizer.normalize_modality(mod)
            normalized_mods.append(normalized)
        
        # 4. Stack into tensor
        stacked = np.stack(normalized_mods, axis=0)  # [4, H, W, D]
        tensor = torch.from_numpy(stacked).unsqueeze(0).float()  # [1, 4, H, W, D]
        
        print(f"   ✅ Stacked tensor: {tensor.shape}")
        print(f"   ✅ Preprocessing complete")
        
        return tensor, brain_mask, affines[0]
    
    def preprocess_batch(self, patient_list):
        """Preprocess multiple patients with progress tracking"""
        results = []
        for patient_id, paths_dict in tqdm(patient_list, desc="Preprocessing Batch"):
            try:
                tensor, mask, affine = self.preprocess_patient(
                    paths_dict['t1'],
                    paths_dict['t1ce'],
                    paths_dict['t2'],
                    paths_dict['flair']
                )
                results.append({
                    'patient_id': patient_id,
                    'tensor': tensor,
                    'mask': mask,
                    'affine': affine,
                    'status': 'success'
                })
            except Exception as e:
                print(f"❌ Error processing {patient_id}: {e}")
                results.append({
                    'patient_id': patient_id,
                    'status': 'failed',
                    'error': str(e)
                })
        
        return results

# Initialize preprocessor
preprocessor = MultimodalPreprocessor(config)
print("\n✅ Multimodal Preprocessor Ready")

In [ ]:
# Section 3.3: Brain Mask Generation
class BrainMaskGenerator:
    """Generate brain mask from MRI volumes using Otsu's method"""
    
    @staticmethod
    def otsu_mask(volume, closing_size=5, opening_size=3):
        """
        Generate brain mask using Otsu's thresholding
        
        Args:
            volume: Input MRI volume
            closing_size: Morphological closing kernel size
            opening_size: Morphological opening kernel size
        
        Returns:
            mask: Binary brain mask [0=background, 1=brain]
        """
        from skimage.filters import threshold_otsu
        from scipy.ndimage import binary_closing, binary_opening
        
        # Otsu thresholding
        threshold = threshold_otsu(volume)
        mask = volume > threshold
        
        # Morphological operations to clean up
        mask = binary_closing(mask, structure=np.ones((closing_size, closing_size, closing_size)))
        mask = binary_opening(mask, structure=np.ones((opening_size, opening_size, opening_size)))
        
        # Keep largest connected component
        labeled, num_features = ndimage.label(mask)
        if num_features > 0:
            sizes = np.bincount(labeled.ravel())
            largest = np.argmax(sizes[1:]) + 1
            mask = labeled == largest
        
        return mask.astype(np.uint8)
    
    @staticmethod
    def common_mask_from_multimodal(modalities):
        """
        Generate common brain mask from multiple modalities
        Uses intersection of individual masks for robustness
        
        Args:
            modalities: List of volume arrays (each is [H, W, D])
        
        Returns:
            common_mask: Binary mask [H, W, D]
        """
        individual_masks = []
        for mod in modalities:
            mask = BrainMaskGenerator.otsu_mask(mod)
            individual_masks.append(mask)
        
        # Intersection of all masks
        common_mask = np.ones_like(individual_masks[0])
        for mask in individual_masks:
            common_mask = common_mask & mask
        
        return common_mask

# Test mask generator
print("✅ Brain Mask Generator initialized")

In [ ]:
# Section 3.2: Adaptive Intensity Normalization
class IntensityNormalizer:
    """Per-modality intensity clipping and z-score normalization"""
    
    def __init__(self, lower_percentile=0.5, upper_percentile=99.5):
        self.lower_pct = lower_percentile
        self.upper_pct = upper_percentile
        self.eps = 1e-6
    
    def adaptive_clip(self, volume):
        """Percentile-based intensity clipping"""
        if volume.max() == 0:
            return volume
        
        nonzero = volume[volume > 0]
        if len(nonzero) == 0:
            return volume
        
        lower = np.percentile(nonzero, self.lower_pct)
        upper = np.percentile(nonzero, self.upper_pct)
        clipped = np.clip(volume, lower, upper)
        return clipped
    
    def z_score_normalize(self, volume, mask=None):
        """Z-score normalization within brain mask"""
        if mask is None:
            mask = volume > 0
        
        if mask.sum() == 0:
            return volume
        
        region = volume[mask]
        mean = region.mean()
        std = region.std()
        
        if std < self.eps:
            normalized = volume - mean
        else:
            normalized = (volume - mean) / std
        
        return normalized
    
    def normalize_modality(self, volume):
        """Complete normalization pipeline for single modality"""
        volume = self.adaptive_clip(volume)
        volume = self.z_score_normalize(volume)
        return volume

# Test normalizer
normalizer = IntensityNormalizer(lower_percentile=0.5, upper_percentile=99.5)
print("✅ Intensity Normalizer initialized")

In [ ]:
# Section 3.1: NIfTI Data Loading
class NIfTILoader:
    """Load and validate NIfTI medical imaging files"""
    
    @staticmethod
    def load_volume(filepath, dtype=np.float32):
        """
        Load NIfTI volume with metadata preservation
        
        Args:
            filepath: Path to .nii or .nii.gz file
            dtype: NumPy data type for output
        
        Returns:
            volume: 3D array [H, W, D]
            affine: 4x4 affine transformation matrix
            header: NIfTI header with metadata
        """
        try:
            nib_img = nib.load(filepath)
            volume = nib_img.get_fdata(dtype=dtype)
            affine = nib_img.affine
            header = nib_img.header
            return volume, affine, header
        except Exception as e:
            print(f"❌ Error loading {filepath}: {e}")
            return None, None, None
    
    @staticmethod
    def save_volume(filepath, volume, affine=None):
        """Save volume as NIfTI file"""
        os.makedirs(Path(filepath).parent, exist_ok=True)
        if affine is None:
            affine = np.eye(4)
        img = nib.Nifti1Image(volume, affine)
        nib.save(img, filepath)
        print(f"✅ Saved volume to {filepath} ({volume.shape})")

# Test NIfTI loader
print("✅ NIfTI Loader initialized")

# Section 3: Data Preprocessing Pipeline

**Preprocessing Steps:**
1. **Load NIfTI volumes** for each modality (T1, T1ce, T2, FLAIR)
2. **Adaptive intensity clipping**: Percentile-based (0.5%-99.5%) to suppress noise
3. **Brain mask generation**: Otsu's method + morphological operations
4. **Z-score normalization**: Per-modality, within brain mask region only
5. **Stacking**: Combine normalized volumes into [4, H, W, D] tensor
6. **Device transfer**: Move to GPU with proper dtype (float32)

**Medical Imaging Best Practices:**
- Each modality normalized independently (different intensity ranges)
- Common brain mask applied to all modalities (prevents alignment issues)
- Statistics computed within brain region only (excludes background noise)
- Z-score used instead of min-max (handles intensity outliers robustly)

In [ ]:
# Section 2.2: Model Loading Functions
class SegmentationModelLoader:
    """Load pre-trained UNet3D segmentation model"""
    
    @staticmethod
    def load(checkpoint_path, device='cuda'):
        """
        Load UNet3D model from checkpoint
        
        Args:
            checkpoint_path: Path to .pth checkpoint file
            device: Target device ('cuda' or 'cpu')
        
        Returns:
            model: Loaded model in eval mode
        """
        print(f"⏳ Loading segmentation model from {checkpoint_path}...")
        
        # Verify file exists
        if not os.path.exists(checkpoint_path):
            raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")
        
        # Load checkpoint
        checkpoint = torch.load(checkpoint_path, map_location=device)
        
        # Extract model state dict
        if isinstance(checkpoint, dict):
            if 'model_state_dict' in checkpoint:
                state_dict = checkpoint['model_state_dict']
                epoch = checkpoint.get('epoch', '?')
            else:
                state_dict = checkpoint
                epoch = '?'
        else:
            state_dict = checkpoint
            epoch = '?'
        
        # Check state dict keys (for model architecture compatibility)
        print(f"   Checkpoint Epoch: {epoch}")
        print(f"   State Dict Keys: {len(state_dict)} parameters")
        
        # Model will be initialized by user (see Section 3)
        return state_dict
    
    @staticmethod
    def validate_checkpoint(checkpoint_path):
        """Validate checkpoint file integrity"""
        try:
            ckpt = torch.load(checkpoint_path, map_location='cpu')
            if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:
                return True, f"Valid checkpoint with {len(ckpt['model_state_dict'])} parameters"
            return True, f"Valid model weights with {len(ckpt)} parameters"
        except Exception as e:
            return False, str(e)

# Test checkpoint validation
seg_checkpoint = config['paths'].get('segmentation_model')
if seg_checkpoint:
    is_valid, msg = SegmentationModelLoader.validate_checkpoint(seg_checkpoint)
    print(f"\n{'✅' if is_valid else '❌'} Segmentation Checkpoint: {msg}")

print("\n📄 Model Loading Framework Ready")

In [ ]:
# Section 2.1: Configuration Management
class PipelineConfig:
    """Centralized configuration for all pipeline components"""
    def __init__(self, config_path="pipeline_config.yaml"):
        if os.path.exists(config_path):
            with open(config_path) as f:
                self.config = yaml.safe_load(f)
            print(f"✅ Loaded config from {config_path}")
        else:
            # Default configuration
            self.config = {
                'paths': {
                    'segmentation_model': 'segmentaion-module/model-weight/final_model_unet.pth',
                    'synthesis_model': 'synthesis-module/model-weight/epoch_118.pth',
                    'output_dir': './output',
                    'log_dir': './logs'
                },
                'preprocessing': {
                    'modalities': ['t1', 't1ce', 't2', 'flair'],
                    'lower_percentile': 0.5,
                    'upper_percentile': 99.5,
                    'normalization': 'z_score',
                    'target_spacing': 1.0
                },
                'segmentation': {
                    'input_channels': 4,
                    'num_classes': 4,
                    'patch_size': 64,
                    'overlap': 0.5
                },
                'reconstruction': {
                    'marching_cubes_level': 0.5,
                    'smooth_iterations': 3,
                    'smooth_lambda': 0.5,
                    'decimation_ratio': 0.5
                }
            }
            print("⚠️  No config file → Using default parameters")
    
    def __getitem__(self, key):
        return self.config.get(key, {})
    
    def save(self, path):
        os.makedirs(Path(path).parent, exist_ok=True)
        with open(path, 'w') as f:
            yaml.dump(self.config, f)
        print(f"✅ Config saved to {path}")

# Load or create configuration
config = PipelineConfig()
print("\n📋 Pipeline Configuration Loaded")
print(f"   Segmentation Model: {config['paths'].get('segmentation_model')}")
print(f"   Modalities: {config['preprocessing'].get('modalities')}")
print(f"   Patch Size: {config['segmentation'].get('patch_size')}")

# Section 2: Module Initialization and Model Loading

**Key Tasks:**
1. Load pre-trained segmentation model (UNet3D with checkpoint)
2. Load optional synthesis model for missing modality recovery
3. Initialize data loaders and preprocessing utilities
4. Configure device management and memory optimization
5. Validate model architecture and weight compatibility

In [ ]:
# 0. ENVIRONMENT & IMPORTS
import os
import sys
import yaml
import json
import torch
import numpy as np
import nibabel as nib
from pathlib import Path
from datetime import datetime
from tqdm import tqdm
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from skimage import measure
import plotly.graph_objects as go
from scipy import ndimage

# GPU Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    
# Suppress warnings
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

# Section 1: System Architecture Overview

The framework is organized into **5 modular components**:

```
Raw Patient Data (4 MRI modalities)
         ↓
[Preprocessing Module] → Intensity clipping, Z-score normalization
         ↓
[Missing Modality Synthesis] → (Optional) Generate missing sequences
         ↓
[Segmentation Module] → UNet3D inference, voxel-wise classification
         ↓
[Post-Processing] → Component cleanup, mask refinement
         ↓
[3D Reconstruction] → Marching cubes, mesh export (OBJ/STL)
         ↓
[Visualization] → Interactive 3D, multi-view rendering
         ↓
[Clinical Reporting] → Dice, HD95, volumetric metrics
```

**Design Principles:**
- **Modular**: Each module can be used independently or as part of the pipeline
- **Configuration-Driven**: All hyperparameters externalized to config files
- **Memory-Efficient**: Sliding window inference for large volumes (155×240×240)
- **Clinically Compatible**: Outputs in standard formats (NIfTI, OBJ, STL, HTML)
- **Graceful Degradation**: System continues even if synthesis fails

# Unified Brain Tumor Analysis Framework
## End-to-End Pipeline: Preprocessing → Synthesis → Segmentation → 3D Reconstruction

**Framework Overview:**
This notebook demonstrates a modular, clinical-grade pipeline for comprehensive brain tumor analysis from multi-modal MRI data. The system integrates:
- **Multi-modal Preprocessing**: Adaptive intensity normalization for T1, T1ce, T2, FLAIR sequences
- **Missing Modality Recovery**: Diffusion-based synthesis for incomplete MRI scans
- **Tumor Segmentation**: UNet3D model producing voxel-wise predictions for ET, TC, WT regions
- **3D Reconstruction**: Marching cubes → mesh generation → surgical planning exports (OBJ/STL)
- **Interactive Visualization**: Hierarchical tumor region rendering with Plotly + matplotlib

**Clinical Use Case:**
Automated brain tumor delineation for surgical planning, radiotherapy dose calculation, and longitudinal treatment monitoring.